In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import plotly.graph_objects as go

### データ読み込み

In [2]:
data = pd.read_parquet('alpha_error_sim_result.parquet')
data['Total_sample_size'] = data['SAMPLE_SIZE_G1'] + data['SAMPLE_SIZE_G2']
data['Variance_ratio'] = (data['SIGMA_G2'] / data['SIGMA_G1']) ** 2
data['Sample_size_ratio'] = data['SAMPLE_SIZE_G2'] / data['SAMPLE_SIZE_G1']
print(data.shape)

(7497, 27)


In [3]:
print(f'Student min sim size: {data["student_valid_n"].min()}')
print(f'Welch min sim size: {data["welch_valid_n"].min()}')

Student min sim size: 1000000
Welch min sim size: 1000000


### 合計サンプルサイズごとにαエラーのヒートマップを作製

In [78]:
def make_heatmap(
    df,
    x_col,
    y_col,
    color_col,
    ci_low_col,
    ci_high_col,
    range_max,
    range_min,
    max_step,
    min_step,
    name,
    significance=0.05
):
    # 変数を絞り込んだデータ作成
    plot_data = df[[x_col, y_col, color_col, ci_low_col, ci_high_col]].copy()
    
    # 信頼区間が名目水準を含む場合はαエラーは名目水準と等しいとする
    plot_data[color_col] = [significance if (low <= significance) & (high >= significance) else value for value, low, high in zip(plot_data[color_col], plot_data[ci_low_col], plot_data[ci_high_col])]
    
    # Pivot
    pivot = plot_data.pivot(index=y_col, columns=x_col, values=color_col)
    
    # Plot
    norm = TwoSlopeNorm(vmin=range_min, vcenter=significance, vmax=range_max)
    lower = np.arange(range_min, significance, min_step)
    upper = np.arange(significance, range_max, max_step)
    fig, ax = plt.subplots(1, 1, figsize=(10, 8), dpi=100)
    mesh = ax.pcolormesh(pivot.columns.values, pivot.index.values, pivot.values, cmap='RdBu_r', norm=norm, shading='nearest')
    cbar = plt.colorbar(mesh)
    cbar.set_ticks(np.concatenate([lower, upper]))
    ax.axvline(1, color='k', linestyle='dashed', linewidth=0.5)
    ax.axhline(1, color='k', linestyle='dashed', linewidth=0.5)
    ax.set_ylim(plot_data[y_col].min(), plot_data[y_col].max())
    ax.set_xscale('log')
    ax.set_yscale('log')
    plt.tight_layout()
    plt.savefig(f'./{name}.png', facecolor='w')
    plt.close()

In [79]:
for total_sample_size in data['Total_sample_size'].unique():

    target_data = data[data['Total_sample_size']==total_sample_size].copy()
    
    make_heatmap(
        df=target_data,
        x_col="Variance_ratio",
        y_col='Sample_size_ratio',
        color_col='student_alpha_error',
        ci_low_col='student_ci_low',
        ci_high_col='student_ci_high',
        range_max=0.1,
        range_min=0,
        max_step=0.01,
        min_step=0.01,
        name=f'alpha-errot-heatmap_student_total-sample-size-{total_sample_size}'
    )
    
    make_heatmap(
        df=target_data,
        x_col="Variance_ratio",
        y_col='Sample_size_ratio',
        color_col='welch_alpha_error',
        ci_low_col='welch_ci_low',
        ci_high_col='welch_ci_high',
        range_max=0.1,
        range_min=0,
        max_step=0.01,
        min_step=0.01,
        name=f'alpha-errot-heatmap_welch_total-sample-size-{total_sample_size}'
    )

### サンプルサイズ比を固定したラインプロット

In [104]:
# サンプルサイズ比=1の場合にstudentとwelchの比較
for total_sample_size in data['Total_sample_size'].unique():

    # Plot用データ作成
    target_data = data[data['Total_sample_size']==total_sample_size].copy()
    target_data = target_data[target_data['Sample_size_ratio']==1].copy()
    target_data.sort_values('Variance_ratio', inplace=True)
    
    # Plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 6), dpi=100)
    ax.scatter(target_data['Variance_ratio'], target_data['student_alpha_error'], color='tab:blue', alpha=0.8)
    ax.plot(target_data['Variance_ratio'], target_data['student_alpha_error'], color='tab:blue', label='Student', alpha=0.8)
    ax.fill_between(target_data['Variance_ratio'], target_data['student_ci_low'], target_data['student_ci_high'], color='tab:blue', alpha=0.4)
    
    ax.scatter(target_data['Variance_ratio'], target_data['welch_alpha_error'], color='tab:orange', alpha=0.8)
    ax.plot(target_data['Variance_ratio'], target_data['welch_alpha_error'], color='tab:orange', label='Welch', alpha=0.8)
    ax.fill_between(target_data['Variance_ratio'], target_data['welch_ci_low'], target_data['welch_ci_high'], color='tab:orange', alpha=0.4)
    
    ax.axhline(0.05, color='k', linestyle='dashed', zorder=1)
    ax.set_xscale('log')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=18)
    ax.set_ylim(0.0485, 0.057)
    plt.tight_layout()
    plt.savefig(f'./alpha-error-lineplot_sample_size_ratio_1_total-sample-size-{total_sample_size}.png', facecolor='w')
    plt.close()

### 3Dプロット

In [4]:
def make_3d_surface(
    df,
    x_col,
    y_col,
    color_col,
    zrange,
    name,
    significance=0.05
):
    # 対数変換した比率を作成
    df[y_col+'_log'] = np.log10(df[y_col])
    df[x_col+'_log'] = np.log10(df[x_col])
    
    # Pivot
    pivot = df.pivot(index=y_col+'_log', columns=x_col+'_log', values=color_col)
    
    # Color scale
    cmin = df[color_col].min()
    cmax = df[color_col].max()
    center = significance
    p = (center - cmin) / (cmax - cmin)
    colorscale = [
        [0.0,        'rgb(5,48,97)'],
        [p*0.5,      'rgb(67,147,195)'],   # 下側の中間（青）
        [p,          'rgb(255,255,255)'],  # 0.05 → 白
        [p+(1-p)*0.5,'rgb(214,96,77)'],    # 上側の中間（赤）
        [1.0,        'rgb(103,0,31)'],
    ]
    
    # significance plane
    Z_plane = np.full_like(pivot.values, significance)
    
    # Plot
    fig = go.Figure(data=[
        go.Surface(
            x=pivot.columns.values,
            y=pivot.index.values,
            z=pivot.values,
            colorscale=colorscale,
            cmin=cmin,
            cmax=cmax,
            colorbar=dict(title='alpha_error')
        ),
        go.Surface(
            x=pivot.columns.values, 
            y=pivot.index.values, 
            z=Z_plane,
            showscale=False,                     # この面のカラーバーは出さない
            opacity=0.3,                         # 半透明にして下の面を透かす
            colorscale=[[0, 'gray'], [1, 'gray']] # 単色（灰色）
        )
    ])
    
    fig.update_layout(
        scene=dict(
            xaxis_title=x_col+'_log',
            yaxis_title=y_col+'_log',
            zaxis_title='alpha_error',
            zaxis=dict(range=zrange),
        ),
        title='alpha_error surface'
    )
    html_str = fig.to_html(include_plotlyjs='cdn')
    with open(f'{name}.html', 'w', encoding='utf-8') as f:
        f.write(html_str)    

In [5]:
for total_sample_size in data['Total_sample_size'].unique():

    target_data = data[data['Total_sample_size']==total_sample_size].copy()
    
    make_3d_surface(
        df=target_data,
        x_col='Variance_ratio',
        y_col='Sample_size_ratio',
        color_col='student_alpha_error',
        zrange=[-0.01, 0.42],
        name=f'alpha-errot-surface_student_total-sample-size-{total_sample_size}',
    )
    
    make_3d_surface(
        df=target_data,
        x_col='Variance_ratio',
        y_col='Sample_size_ratio',
        color_col='welch_alpha_error',
        zrange=[0.045, 0.08],
        name=f'alpha-errot-surface_welch_total-sample-size-{total_sample_size}',
    )